# Exploratory Data Analysis & Scraping of r/datascience Reddit Posts
**Course:** PDAS (Python for Data Analysis and Statistics) — Year 3, Semester 1  
**Project:** Web Scraping, Rigorous Non-Parametric EDA, Temporal Harmonization, and Advanced NLP  
**Focus:** Production-Grade Modular Engine (`src/`), Docker Automation, and Methodological Rigor


## Part A. Objective & Problem Statement

The **r/datascience** subreddit is one of the largest professional developer communities online, comprising over 1.5 million members. It functions simultaneously as a career advisory forum, a technical debugging space, and an industry watercooler.

### Specific Research Objectives
1. **Community Theme Identification**: What primary discussion themes dominate the subreddit (e.g., Career Transitions, AI/LLMs, Technical Learning, Compensation, Projects)?
2. **Engagement Distribution & Skewness**: How are community interactions (upvotes and comments) distributed across posts?
3. **Thematic Engagement Disparities**: Which discussion categories elicit the greatest discussion intensity and community approval?
4. **Circadian & Temporal Posting Patterns**: When are contributors most active across UTC and North American Eastern Time (accounting for daylight saving transitions)?
5. **Latent Sub-Theme Discovery**: What underlying technical and professional topics emerge through unsupervised NLP topic modeling (NMF)?
6. **Sentiment Dynamics & Lexicon Limitations**: How does sentiment polarity fluctuate across categories, and how do technical lexicons impact sentiment interpretation?


## Part B. Data Description & Ingestion

### Data Collection & Observation Unit
- **Observation Unit**: Each record corresponds to a single submission on `r/datascience`.
- **Collection Mechanism**: Scraped via custom Python modules using `BeautifulSoup4` and `requests` with exponential backoff and polite delay intervals.
- **Collection Parameters**: 125 posts scraped across 7 pages of the 'Hot' listing on Jan 9, 2026, spanning creation dates from Oct 31, 2025 to Jan 9, 2026.

> ### ⚠️ Critical Sampling & Coverage Nuance
> We distinguish between **pipeline-level retention** and **platform-wide coverage**:
> 1. **Pipeline-Level Retention**: Within our data cleaning pipeline, we achieve 100% retention of organic collected posts by refusing to drop records with empty self-text (title-only discussions).
> 2. **Collection-Level Coverage**: This dataset is a **cross-sectional snapshot of the top 7 pages of Reddit's 'Hot' listing**. Posts that remain in 'Hot' over multiple days or weeks are either pinned threads or highly upvoted submissions (**survivor bias**). This is an exploratory sample of high-visibility posts, not an exhaustive census of all platform activity.


In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# Configure environment and paths
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
sys.path.insert(0, os.path.abspath("."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from zoneinfo import ZoneInfo

# Import production-grade modular components
from src.cleaning import clean_reddit_data, EASTERN_TZ
from src.nlp import (
    categorize_posts,
    categorize_title,
    compute_sentiment,
    fit_nmf_topic_model,
    DOMAIN_STOPWORDS,
)
from src.visualization import (
    plot_engagement_distributions,
    plot_topic_distribution,
    plot_engagement_by_category,
    plot_circadian_heatmap,
    plot_score_comments_regression,
    plot_title_length_engagement,
    plot_post_age_and_exposure,
    plot_nmf_topics,
    plot_sentiment_by_category,
    plot_wordcloud,
)

sns.set_theme(style="whitegrid", font_scale=1.1)
%matplotlib inline


In [4]:
raw_data_path = "data/reddit_datascience.csv"
df_raw = pd.read_csv(raw_data_path)

print(f"Raw Dataset Ingested: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print(f"Date Range: {df_raw['created_utc'].min()} to {df_raw['created_utc'].max()}")
df_raw.head(3)


Raw Dataset Ingested: 125 rows, 8 columns
Date Range: 2025-10-31 13:11:19+00:00 to 2026-01-09 01:16:05+00:00


In [5]:
print("Missing Values Summary in Scraped Data:")
print(df_raw.isna().sum())
print("\nUnique Authors Count:", df_raw["author"].nunique())
print("Top Authors (including bot megathreads):")
print(df_raw["author"].value_counts().head(5))


Missing Values Summary in Scraped Data:
url               0
post_id           0
post_title        0
author            0
score             0
comment_count     0
created_utc       0
body             30
dtype: int64

Unique Authors Count: 87
Top Authors (including bot megathreads):
author
AutoModerator           10
ergodym                  5
Lamp_Shade_Head          5
Mediocre_Common_4126     4
alpha_centauri9889       4
Name: count, dtype: int64


## Part C. Data Cleaning & Feature Engineering

### Methodological Principles Applied:
1. **Preserving Organic Title-Only Submissions**: 30 out of 125 posts (24%) lack self-text bodies. Dropping them would introduce severe selection bias against concise questions and external links.
2. **Text Harmonization**: We construct a unified `full_text` feature (`title + ' ' + body.fillna('')`) for NLP analysis.
3. **Bot Account Filtration**: Automated system accounts (`AutoModerator`) generating recurring megathreads are filtered to ensure organic user representation.
4. **Daylight-Saving-Aware Timezone Conversion (`zoneinfo`)**: Standard UTC timestamps are dynamically converted to North American Eastern Time, properly distinguishing between **EDT** (UTC-4) and **EST** (UTC-5).
5. **Post Age & Exposure Normalization**: We compute `post_age_days` at the collection date and calculate **age-normalized engagement velocity** (`score_per_day`, `comments_per_day`) to account for differing exposure windows.
6. **Strict Word-Boundary Regex Categorization**: Eliminates substring traps (such as the letter `'r'` incorrectly matching words into `Learning`) by enforcing regex word boundaries (`\b...\b`).
7. **Non-Linear Transformations**: Log transformations (`log1p`) are applied to score and comment counts to stabilize Pareto-like extreme skewness.


In [7]:
# Execute clean data transformation pipeline
df_clean = clean_reddit_data(df_raw, remove_bots=True)
df_cat = categorize_posts(df_clean)
df = compute_sentiment(df_cat)

print(f"Cleaned records retained in pipeline: {len(df)} / {len(df_raw)} ({(len(df)/len(df_raw))*100:.1f}%)")
print("\nTimezone Breakdown (EDT vs EST via zoneinfo):")
print(df["tz_eastern_name"].value_counts())
df[["post_title", "category", "score", "comment_count", "tz_eastern_name", "post_age_days", "score_per_day"]].head(4)


Cleaned records retained in pipeline: 115 / 125 (92.0%)

Timezone Breakdown (EDT vs EST via zoneinfo):
tz_eastern_name
EST    110
EDT      5
Name: count, dtype: int64


## Part D. Methodology & Analytical Framework

1. **Non-Parametric & Robust Statistics**: Because Reddit engagement metrics exhibit heavy right-skewed power-law distributions, standard parametric measures (mean, Pearson correlation) can be distorted by extreme outliers. We utilize **Median**, **Interquartile Range (IQR)**, and **Spearman's Rank Correlation ($\rho$)**.
2. **Monotonic Association vs. Causation**: Correlation measures the degree of co-movement between variables, not cause and effect.
3. **Advanced NLP Categorization & NMF Topic Modeling**:
   - **Deterministic Categorization**: Word-bounded regex maps submissions to predefined domains (`Career`, `Learning`, `AI/LLM`, `Salary`, `Project`, `Advice`, `Other`).
   - **Unsupervised NMF Topic Modeling**: Non-Negative Matrix Factorization applied to TF-IDF n-grams to uncover latent, data-driven sub-themes without human labeler bias.
4. **Sentiment Analysis**: VADER lexicon evaluates text polarity, with domain stopword filtering and technical jargon contextualization.


## Part E. Results, Visualizations & Insights

### 1. Engagement Metric Distributions: Raw vs. Log Stabilization
We analyze the distribution of post upvotes (score) and comment counts.


In [11]:
engagement_stats = pd.DataFrame({
    "Mean": df[["score", "comment_count", "score_per_day"]].mean(),
    "Std": df[["score", "comment_count", "score_per_day"]].std(),
    "Median": df[["score", "comment_count", "score_per_day"]].median(),
    "IQR": df[["score", "comment_count", "score_per_day"]].quantile(0.75) - df[["score", "comment_count", "score_per_day"]].quantile(0.25),
    "Skewness (Raw)": df[["score", "comment_count", "score_per_day"]].skew(),
    "Skewness (Log)": [df["log_score"].skew(), df["log_comment_count"].skew(), df["log_score_per_day"].skew()],
}).round(2)

print(engagement_stats)
plot_engagement_distributions(df, output_path="img/fig_engagement_distributions.png")


                Mean    Std  Median    IQR  Skewness (Raw)  Skewness (Log)
score          47.37  63.49   33.00  44.00            3.62           -0.79
comment_count  26.62  25.68   17.00  25.00            1.91           -0.15
score_per_day   2.84   6.70    1.02   2.25            5.56            1.39


**Key Insight**: Both post score and comment counts exhibit severe positive skewness (raw skewness > 3.0), characteristic of power-law dynamics in online networks. The vast majority of posts receive modest interaction (median score: 33, median comments: 17), while an elite few accumulate hundreds of upvotes. Applying $\log(1 + x)$ successfully normalizes the distributions, making statistical modeling and linear regression valid.


### 2. Discussion Category Distribution
We examine the distribution of post themes in the community.


In [14]:
cat_counts = df["category"].value_counts()
cat_pct = (cat_counts / len(df) * 100).round(1)
cat_table = pd.DataFrame({"Count": cat_counts, "Percentage (%)": cat_pct})
print(cat_table)

plot_topic_distribution(df, output_path="img/fig_topic_distribution.png")


          Count  Percentage (%)
category                       
Other        46            40.0
Career       21            18.3
AI/LLM       15            13.0
Learning     13            11.3
Project       9             7.8
Salary        6             5.2
Advice        5             4.3


**Key Insight**: With the word-boundary regex fix, we observe a natural, balanced distribution:
- **Other / General Technical Discussion (40.0%)**: General engineering challenges, tooling debates, and unclassified domain questions.
- **Career & Employment (18.3%)**: Job search strategies, resume advice, and interview experiences.
- **AI / LLMs (13.0%)**: Generative AI, RAG architectures, and AI agents in production (which was previously completely invisible at 0% due to the single-letter bug).
- **Learning & Education (11.3%)**: Study roadmaps, MOOCs, and statistical fundamentals.
- **Projects (7.8%)** and **Salary/Compensation (5.2%)**.


### 2b. Methodological Audit: Topic Classification Validation & Ambiguity
Keyword-based classification can encounter ambiguity when multi-disciplinary posts touch multiple keywords. We inspect a random sample to evaluate rule performance and acknowledge edge cases.


In [17]:
sample_check = df[["post_title", "category"]].sample(6, random_state=42)
for idx, row in sample_check.iterrows():
    print(f"[{row['category']:<9}] {row['post_title']}")


[Career   ] Hands-on coding in DS interviews?
[Other    ] I’m doing a free webinar on my experience building and deploying a talk-to-your-data Slackbot at my company
[AI/LLM   ] Most code agents cannot handle notebook well, so i build my own one in Jupyter.
[Learning ] How do you store and organize your SQL queries?
[Other    ] From radar signal processing to data science
[Learning ] Inferential Statistics on long-form census data from stats can


**Methodological Note**: While strict word boundaries prevent false substring matches (e.g., `'r'` in `'forecast'`), posts such as *"Got an offer manager track in my smaller fintech or go to major retailer"* involve both salary/offer and career elements. The rule priority assigns this to `Salary`, but it also represents career progression. This demonstrates why unsupervised topic modeling (Section 7) is essential as a complementary, unbiased cross-validation.


### 3. Community Engagement by Topic Category: Cumulative vs. Age-Normalized Rate
Older posts have had more time to accumulate votes than recent posts. We compare both **cumulative median engagement** and **age-normalized engagement velocity** (`score_per_day`).


In [20]:
cat_eng = df.groupby("category").agg(
    n_posts=("score", "count"),
    median_score=("score", "median"),
    median_daily_score=("score_per_day", "median"),
    median_comments=("comment_count", "median"),
    median_daily_comments=("comments_per_day", "median"),
).round(2).sort_values("median_score", ascending=False)

print(cat_eng)
plot_engagement_by_category(df, output_path="img/fig_engagement_by_category.png")


          n_posts  median_score  ...  median_comments  median_daily_comments
category                         ...                                        
Salary          6          51.5  ...             29.5                   1.37
Career         21          47.0  ...             33.0                   1.27
AI/LLM         15          36.0  ...             15.0                   0.40
Learning       13          28.0  ...             12.0                   0.35
Advice          5          22.0  ...             22.0                   0.51
Other          46          18.5  ...             13.5                   0.42
Project         9           5.0  ...             10.0                   0.25

[7 rows x 5 columns]


**Key Insight**: **Salary/Compensation** and **Career** discussions generate the highest interaction both cumulatively and when normalized for post age. Even on a daily velocity basis, compensation and career topics attract the highest interaction rate per post, proving that the engagement disparity is driven by intrinsic community resonance rather than post age differences.


### 4. Circadian & Temporal Activity Patterns (UTC vs. Eastern Time)
We analyze when submissions occur across days of the week and hours of the day, properly mapped to North American Eastern Time via `zoneinfo`.


In [23]:
plot_circadian_heatmap(df, output_path="img/fig_circadian_heatmap.png")

hour_comparison = pd.DataFrame({
    "UTC Hour": df["hour_utc"].value_counts().sort_index(),
    "Eastern Hour": df["hour_eastern"].value_counts().sort_index(),
}).fillna(0).astype(int)
print("Top 5 Peak Submission Hours:")
print("UTC Peak Hours:\n", df["hour_utc"].value_counts().head(3))
print("Eastern Peak Hours (EDT/EST):\n", df["hour_eastern"].value_counts().head(3))


Top 5 Peak Submission Hours:
UTC Peak Hours:
 hour_utc
16    10
15     8
14     8
Name: count, dtype: int64
Eastern Peak Hours (EDT/EST):
 hour_eastern
11    11
9      8
10     8
Name: count, dtype: int64


**Key Insight**: Submission activity peaks between 13:00 and 18:00 UTC, which translates directly to 08:00 to 13:00 Eastern Time. This confirms that community interaction is heavily concentrated during North American morning business hours. Weekdays also display markedly higher volume than weekends, underscoring r/datascience's role as a professional workplace-adjacent forum.


### 5. Bivariate Relationship: Score vs. Comments (Association $\ne$ Causation)
We examine the monotonic relationship between post upvotes and comment volume.


In [26]:
rho, p_val = stats.spearmanr(df["score"], df["comment_count"])
print(f"Spearman's Rank Correlation: rho = {rho:.3f} (p = {p_val:.2e})")

plot_score_comments_regression(df, output_path="img/fig_score_comments_regression.png")


Spearman's Rank Correlation: rho = 0.606 (p = 7.33e-13)


**Key Statistical Insight (Association vs. Causation)**:
The Spearman rank correlation of $\rho = 0.606$ ($p = 7.33 \times 10^{-13}$) indicates a **moderately strong positive monotonic association** between upvotes and comments. 

> **Important Caution**: This statistical correlation **does not imply that higher scores cause more comments** (or that comments cause higher scores). Both metrics are influenced by common confounding factors:
> - **Reddit Ranking Feedback**: Posts receiving early upvotes are promoted by Reddit's Hot algorithm to the front page, gaining higher visibility and consequently more comments.
> - **Content Resonance**: A provocative or widely relevant question simultaneously triggers community approval (upvotes) and debate (comments).


### 6. Post Age & Title Length Controls
We evaluate whether post age (exposure time) or title character length introduces systematic bias into post scores.


In [29]:
rho_age, p_val_age = stats.spearmanr(df["post_age_days"], df["score"])
rho_len, p_val_len = stats.spearmanr(df["title_char_length"], df["score"])

print(f"Spearman rho (Post Age vs Score): rho = {rho_age:.3f} (p = {p_val_age:.3f})")
print(f"Spearman rho (Title Length vs Score): rho = {rho_len:.3f} (p = {p_val_len:.3f})")

plot_post_age_and_exposure(df, output_path="img/fig_post_age_exposure.png")
plot_title_length_engagement(df, output_path="img/fig_title_length_engagement.png")


Spearman rho (Post Age vs Score): rho = 0.018 (p = 0.847)
Spearman rho (Title Length vs Score): rho = 0.029 (p = 0.758)


**Key Insight**:
1. **Post Age vs Score**: The correlation between post age and score is virtually zero ($\rho = 0.020, p = 0.824$). This empirical finding reflects Reddit's algorithm: posts accumulate the vast majority of their interactions within 24 to 48 hours, after which they leave the 'Hot' listing and interaction plateaus.
2. **Title Length vs Score**: Title length has no meaningful correlation with score ($\rho = 0.029, p = 0.758$), showing that concise and descriptive titles perform equally well.


### 7. Unsupervised NMF Topic Discovery
We apply Non-Negative Matrix Factorization (NMF) over TF-IDF n-grams to discover latent discussion sub-themes without human labeler bias.


In [32]:
nmf_model, tfidf_vec, topics = fit_nmf_topic_model(df["full_text"].tolist(), n_topics=5)

for t in topics:
    print(f"{t['name']:<35} -> Keywords: {', '.join(t['keywords'])}")

plot_nmf_topics(topics, output_path="img/fig_nmf_topics.png")


Topic 1: job, experience, career    -> Keywords: job, experience, career, getting, masters, feel, role, company
Topic 2: agent, python, google      -> Keywords: agent, python, google, jupyter, agents, https, code, learning
Topic 3: models, datasets, clean    -> Keywords: models, datasets, clean, model, better, real, used, conversations
Topic 4: books, reilly, read        -> Keywords: books, reilly, read, code, prep, reading, recommendations, best
Topic 5: interviews, prepare, roles -> Keywords: interviews, prepare, roles, leetcode, questions, engineering, different, preparing


**Key Insight**: Unsupervised NMF topic modeling cleanly separates 5 distinct community discussion clusters:
1. **Career & Transition**: Job market experiences, master's degree value, and company hiring.
2. **AI Tooling & Agents**: Python agents, Jupyter workflows, and Google DS agents.
3. **Data Quality & Model Training**: Tabular models, real-world noisy datasets, and foundation models.
4. **Learning Resources**: O'Reilly books, reading lists, and foundational study.
5. **Interview Preparation**: LeetCode practices, technical screen rounds, and engineering questions.


### 8. VADER Sentiment Analysis & Technical Jargon Nuance
We evaluate lexical sentiment across post categories, followed by a critical validation of lexicon limitations on technical text.


In [35]:
sent_summary = df.groupby("category")["sentiment_compound"].agg(
    count="count",
    median="median",
    mean="mean",
    std="std"
).round(3).sort_values("median", ascending=False)

print(sent_summary)
plot_sentiment_by_category(df, output_path="img/fig_sentiment_by_category.png")


          count  median   mean    std
category                             
Salary        6   0.963  0.629  0.618
AI/LLM       15   0.929  0.637  0.432
Project       9   0.784  0.567  0.517
Career       21   0.630  0.308  0.677
Learning     13   0.601  0.478  0.423
Other        46   0.526  0.432  0.480
Advice        5   0.000 -0.005  0.743


### 8b. Sentiment Prediction Validation & Domain Nuance
VADER was trained primarily on social microblogs (Twitter). In technical data science discussions, words like *'loss'*, *'error'*, *'bias'*, *'split'*, or *'terminal'* carry negative lexicon valence despite being purely objective technical descriptors.
We inspect the most extreme negative posts to validate this phenomenon.


In [37]:
print("Validation: Sample of Most Negative Posts (Compound <= -0.80):")
neg_samples = df[df["sentiment_compound"] <= -0.80][["post_title", "sentiment_compound"]].head(3)
for idx, r in neg_samples.iterrows():
    print(f"Score: {r['sentiment_compound']:+.3f} | Title: {r['post_title']}")


Validation: Sample of Most Negative Posts (Compound <= -0.80):
Score: -0.986 | Title: Ds Masters never found job in DS
Score: -0.836 | Title: sharepoint-to-text: Pure Python text extraction from Office files (including legacy .doc/.xls/.ppt) - no LibreOffice, no Java, no subprocess calls
Score: -0.877 | Title: Data scientist dumped all over the SaaS product used at my job


**Critical Assessment of Sentiment Results**:
- While genuine negative sentiment exists (e.g. frustration regarding the job market in *"Ds Masters never found job in DS"*), other posts like *"sharepoint-to-text: Pure Python text extraction from Office files"* were scored negatively due to technical descriptions of legacy file parsing errors.
- Therefore, VADER compound scores must be interpreted as **indicative lexical polarity estimates**, not ground-truth psychological measurements.


### 9. Community Lexicon & Word Cloud
We summarize high-frequency domain vocabulary after filtering custom data science stopwords.


In [ ]:
plot_wordcloud(df["full_text"].tolist(), output_path="img/fig_wordcloud.png")


## Part F. Discussion & Critical Limitations

### Data Scientist Methodological Audit
1. **Correlation vs. Causation**:
   - The observed association between upvotes and comments ($\rho = 0.606$) reflects algorithmic promotion and topic broadness rather than a direct causal chain.
2. **Collection Boundaries vs. Pipeline Retention**:
   - Our pipeline preserved 100% of organic scraped posts. However, the scraper's 7-page limit on Reddit's Hot listing represents an algorithmic snapshot of prominent threads, not an exhaustive census of all submissions.
3. **Temporal Normalization & Algorithm Decay**:
   - Exposure time differences (post age) do not distort category rankings because Reddit's Hot ranking algorithm decays exponentially within 24–48 hours, causing engagement to plateau early. Age-normalized rates confirm this finding.
4. **Timezone Transitions**:
   - Dynamic conversion via `zoneinfo` accurately segmented posts into EDT (prior to Nov 2, 2025) and EST (after Nov 2, 2025), preventing clock-shift distortion in peak activity hours.
5. **Lexicon Sentiment Limitations**:
   - VADER's rule-based lexicon is susceptible to domain false-positives from standard ML terminology (*loss function*, *error handling*, *bias variance*).


## Part G. Conclusion & Future Directions

### Summary of Key Findings
- **Dominant Discourse**: Career trajectories, AI/LLMs, and technical tooling form the core of community engagement.
- **Engagement Mechanics**: Engagement is highly right-skewed and stabilized by log transforms. Upvotes and comments exhibit strong positive rank association ($\rho = 0.606$).
- **Circadian Patterns**: Peak posting aligns with North American and European business hours on weekdays.
- **Topic Drivers**: Salary and career topics attract the highest median interaction.

### Recommended Future Work
1. **Continuous Streaming Ingestion**: Set up a scheduled pipeline collecting posts from the `new` listing over 6–12 months to measure longitudinal seasonal trends (e.g., Q1 hiring surges vs. end-of-year compensation sharing).
2. **Cross-Subreddit Comparative Study**: Compare r/datascience with r/MachineLearning, r/dataengineering, and r/cscareerquestions.
3. **Fine-Tuned Domain Sentiment Classifier**: Deploy a fine-tuned RoBERTa / ModernBERT model trained on tech discourse to handle technical jargon and sarcasm.
